# Fashion-MNIST: Transaction Construction (TC0–TC7)

Notebook này chạy cùng một pipeline TC0–TC7 theo cấu hình được chọn. Mặc định là smoke (100 train + 20 validation); có thể chọn full bằng biến môi trường `FASHION_TC_CONFIG=configs/tc_full.toml`. Cách kích hoạt project, kernel và thứ tự các cell không đổi; config quyết định kích thước dữ liệu, K/seed, chế độ chạy và output.

Chạy từ đầu bằng kernel mới. Mỗi cell TC gọi một bước trong `src/applications/FashionTC.jl` và hiển thị tóm tắt.

## 1. Tìm project root và kích hoạt môi trường

Tìm `Project.toml` từ thư mục hiện tại hoặc thư mục cha để notebook chạy được dù Jupyter mở từ repo root hay thư mục notebook.


In [21]:
using Pkg
Pkg.instantiate()

function find_repo_root(start = pwd())
    current = abspath(start)
    while true
        isfile(joinpath(current, "Project.toml")) && return current
        parent = dirname(current)
        parent == current && error("Không tìm thấy Project.toml từ $start")
        current = parent
    end
end

const REPO_ROOT = find_repo_root()
Pkg.activate(REPO_ROOT)
println("Project: ", Base.active_project())


Precompiling packages...
   1.4 s  ✓ EclatFIM
  1 dependency successfully precompiled in 2 seconds. 136 already precompiled.
  7 dependencies precompiled but different versions are currently loaded (Clustering, IJulia, NearestNeighbors, PrecompileTools and StaticArrays, and 2 more). Restart julia to access the new versions. Otherwise, 7 dependents of these packages may trigger further precompilation to work with the unexpected versions.
  Activating 

Project: /run/media/fernir/Data/Documents/HCMUS/SEMESTER7/DataMining/TeamProject-FrequentItemsetMining/Project.toml


project at `/run/media/fernir/Data/Documents/HCMUS/SEMESTER7/DataMining/TeamProject-FrequentItemsetMining`


## 2. Đọc cấu hình đã chọn

Mặc định notebook đọc `configs/tc_smoke.toml`. Để dùng cấu hình full, đặt `FASHION_TC_CONFIG=configs/tc_full.toml` trước khi mở/chạy Jupyter. Hai cấu hình đi qua cùng các cell TC0–TC7.

In [22]:
include(joinpath(REPO_ROOT, "src", "EclatFIM.jl"))

const TC = EclatFIM.FashionTC
const CONFIG_RELATIVE_PATH = get(ENV, "FASHION_TC_CONFIG", "configs/tc_full.toml")
const CONFIG_PATH = normpath(joinpath(REPO_ROOT, CONFIG_RELATIVE_PATH))
const CONFIG = TC.read_config(CONFIG_PATH)

println("Config: ", CONFIG_RELATIVE_PATH)
println("Run ID: ", CONFIG["run"]["run_id"])
println("Chế độ: ", CONFIG["run"]["mode"])

Config: configs/tc_full.toml
Run ID: FashionMNIST_T2_benchmark
Chế độ: full


## Chuẩn bị phiên chạy

Tạo trạng thái dùng chung cho notebook và lấy output path từ config. Hàm này áp dụng kiểm tra an toàn phù hợp với mode; nó không tự chọn riêng một pipeline smoke/full.

In [26]:
const DATA_DIR = joinpath(REPO_ROOT, "data", "raw", "fashion-mnist")
const OUTPUT_DIR = normpath(joinpath(REPO_ROOT, CONFIG["run"]["output_root"]))

run_state = TC.prepare_run(CONFIG_PATH; data_dir=DATA_DIR, output_root=OUTPUT_DIR)
println("Output: ", OUTPUT_DIR)

Output: /run/media/fernir/Data/Documents/HCMUS/SEMESTER7/DataMining/TeamProject-FrequentItemsetMining/results/fashion/tc/benchmark


## TC0 — Fit ngưỡng từ train

Kiểm tra checksum archive, nạp split gốc và lấy ảnh theo config. Smoke chọn mẫu cân bằng cố định; full dùng toàn bộ train/test. Q10+ luôn được fit trên train của mode đang chọn, sau đó ngưỡng đó dùng chung cho validation.

In [27]:
tc0_result = TC.run_tc0!(run_state)
display(tc0_result)

(tc = "TC0", label = "Fit ngưỡng train-only", summary = (train_images = 60000, validation_images = 10000, tau = 29.0))

## TC1 — Trích patch 4×4

Chia ảnh thành patch theo thứ tự hàng rồi cột. Smoke kiểm tra ghép patch khôi phục đúng ảnh; full dùng cùng kích thước patch và tính toán theo toàn bộ split.

In [28]:
tc1_result = TC.run_tc1!(run_state)
display(tc1_result)

(tc = "TC1", label = "Trích patch 4×4", summary = (train_patches = 2940000, validation_patches = 490000, reconstruction = "không chạy cho full"))

## TC2 — Lọc patch

Dùng tau của TC0 và alpha trong config để chọn patch. Kết quả gồm số patch giữ/loại theo ảnh và tỉ lệ patch trùng chính xác sau lọc, tính riêng train/validation theo công thức `1 - N_unique / N_total`. Hai patch trùng khi toàn bộ pixel UInt8 giống nhau. CSV `patch_duplicate_summary.csv` lưu số tổng, số unique, số bản trùng và tỉ lệ này.

In [29]:
tc2_result = TC.run_tc2!(run_state)
display(tc2_result)

(tc = "TC2", label = "Lọc patch T2", summary = (train_kept = 1836369, train_discarded = 1103631, validation_kept = 307377, validation_discarded = 182623, train_total_patches = 1836369, train_unique_patches = 1826507, train_duplicate_patches = 9862, train_duplicate_rate = 0.005370380353839521, validation_total_patches = 307377, validation_unique_patches = 306530, validation_duplicate_patches = 847, validation_duplicate_rate = 0.002755573774225084))

## TC3 — Tạo descriptor

Mỗi patch được giữ chuyển thành raw-pixel vector Float32 đã chuẩn hóa. Smoke và full cùng dùng cách biểu diễn, chỉ khác số ảnh/descriptor được xử lý.

In [19]:
tc3_result = TC.run_tc3!(run_state)
display(tc3_result)

(tc = "TC3", label = "Tạo raw-pixel descriptor", summary = (dimension = 16, train_descriptors = 1836369, validation_descriptors = 307377, values = "Float32 trong [0,1]"))

## TC4 — Fit codebook K-means

Fit chỉ trên descriptor train. Smoke chạy một cấu hình K/seed để kiểm tra luồng; full lấy toàn bộ K và seed từ config và chạy tuần tự.

In [20]:
tc4_result = TC.run_tc4!(run_state)
display(tc4_result)

┌ Warning: The clustering cost increased at iteration #40
└ @ Clustering ~/.julia/packages/Clustering/M6mjF/src/kmeans.jl:191
┌ Warning: The clustering cost increased at iteration #26
└ @ Clustering ~/.julia/packages/Clustering/M6mjF/src/kmeans.jl:191
┌ Warning: The clustering cost increased at iteration #27
└ @ Clustering ~/.julia/packages/Clustering/M6mjF/src/kmeans.jl:191
┌ Warning: The clustering cost increased at iteration #28
└ @ Clustering ~/.julia/packages/Clustering/M6mjF/src/kmeans.jl:191
┌ Warning: The clustering cost increased at iteration #29
└ @ Clustering ~/.julia/packages/Clustering/M6mjF/src/kmeans.jl:191
┌ Warning: The clustering cost increased at iteration #30
└ @ Clustering ~/.julia/packages/Clustering/M6mjF/src/kmeans.jl:191
┌ Warning: The clustering cost increased at iteration #31
└ @ Clustering ~/.julia/packages/Clustering/M6mjF/src/kmeans.jl:191
┌ Warning: The clustering cost increased at iteration #32
└ @ Clustering ~/.julia/packages/Clustering/M6mjF/src/kmeans

LoadError: InterruptException:

## TC5 — Gán visual word

Gán train và validation bằng codebook tương ứng. Chunk size hoặc batch size được đọc từ config; không cập nhật tâm cụm ở bước này.

In [9]:
tc5_result = TC.run_tc5!(run_state)
display(tc5_result)

(tc = "TC5", label = "Gán visual words", summary = NamedTuple[(run_id = "T2_K8_seed42", train_assignments = 3006, validation_assignments = 621, chunk_size = 128)])

## TC6 — Tạo transaction BoVW

Gom word ID theo ảnh, loại trùng và sắp xếp. Mỗi ảnh vẫn có một transaction logic; metadata TID giữ liên kết về split và image ID.

In [10]:
tc6_result = TC.run_tc6!(run_state)
display(tc6_result)

(tc = "TC6", label = "Tạo transaction BoVW", summary = NamedTuple[(run_id = "T2_K8_seed42", train_transactions = 100, validation_transactions = 20, empty_train = 0, empty_validation = 0)])

## TC7 — Xuất và kiểm tra artifacts

Smoke reload artifacts, kiểm tra fixture và chạy một Eclat pilot. Full xác minh số transaction/assignment theo từng cấu hình và ghi benchmark summary. Cả hai mode đi qua cùng cell TC7.

In [11]:
tc7_result = TC.run_tc7!(run_state; notebook_fresh_kernel=true)
display(tc7_result.summary)

report_name = CONFIG["run"]["mode"] == "smoke" ? "smoke_test_report.json" : "full_run_report.toml"
println("Report: ", joinpath(OUTPUT_DIR, report_name))

Fresh-process artifact reload PASS: codebook, assignments, descriptors, 120 transactions


Dict{String, Any} with 21 entries:
  "validation_patches_before_filter" => 980
  "kmeans_k"                         => 8
  "pipeline_status"                  => "PASS"
  "train_patches_before_filter"      => 4900
  "notebook_fresh_kernel"            => "PASS"
  "train_images"                     => 100
  "train_transactions"               => 100
  "config_path"                      => "/run/media/fernir/Data/Documents/HCMUS…
  "checks"                           => Dict("tc1_patch_reconstruction"=>"PASS"…
  "kmeans_fit_count"                 => 1
  "elapsed_seconds"                  => 21.4749
  "effective_config_path"            => "/run/media/fernir/Data/Documents/HCMUS…
  "mode"                             => "smoke"
  "train_retained_descriptors"       => 3006
  "transaction_count"                => 120
  "run_id"                           => "T2_K8_seed42"
  "status"                           => "PASS"
  "validation_retained_descriptors"  => 621
  "validation_images"               

Report: /run/media/fernir/Data/Documents/HCMUS/SEMESTER7/DataMining/TeamProject-FrequentItemsetMining/results/fashion/tc/smoke/T2_K8_seed42/smoke_test_report.json